# 19.25 怎樣把判斷器接成可用的裝置服務？


主錄音室要錄音，許可與就緒都已確認；同一份材料還列出播放台、封存NAS和備援錄音室。我們希望模型回答三件事：主錄音室該做哪個動作、它的必要資訊是否齊全、按固定規準可得幾分。如果把就緒改成「不知道」，動作與評分就應拒答。這是本節產品C-v3要完成的有限工作，不需要把任意自由文字都交給它。

[19.23](https://birdhackor.github.io/tiny-perceptron-vlm/19.23.html)的C-v2已教共享材料與候選隔離，也保留未達標的實測。本節沿用那個計算分工，改成明確的裝置介面、分筆記憶體及重新訓練的模型。當前裝置決策推薦使用本節C-v3；舊文字C、B的Auto effort及C-v2路由實驗保留各自入口與原始成績。

**先把服務規則與輸入說清楚。** `record/play/save`表示錄音／播放／儲存；`permission`是有無許可，`ready`是是否就緒。JSON的`true/false`是已知的是／否，`null`才是不知道。一次提交3–6個裝置，每個都有不重複的名字；`target`指出這次問誰。下面就是可直接交給公開入口的完整請求：

```json
{
  "devices": [
    {"id": "主錄音室", "task": "record", "permission": true, "ready": true},
    {"id": "播放台", "task": "play", "permission": true, "ready": false},
    {"id": "封存NAS", "task": "save", "permission": false, "ready": true},
    {"id": "備援錄音室", "task": "record", "permission": false, "ready": false}
  ],
  "target": "主錄音室",
  "kind": "all",
  "language": "zh"
}
```

`kind=all`請服務各答一題，`language=zh`讓服務產生固定的中文內部題目；也可選`en`。它不是讓模型翻譯任意規則。三題都查同一份材料，但需要的資訊與答案不同：

| 題型 | 本服務事前固定的規則 | 主錄音室本筆正解 |
| --- | --- | --- |
| Choice選動作 | 許可與就緒都須已知；無許可先stop，否則未就緒wait，否則做task；正確動作不在候選也答none | record |
| info資訊充分性 | record要兩欄；play只要許可；save只要就緒。已知false也算完整 | yes |
| rubric固定評分 | 兩欄都須已知；record數兩欄有幾個true；play在許可為true時給2，否則按就緒給0或1；save在就緒為true時給2，否則按許可給0或1 | 2 |

因此只把主錄音室的`ready`改成`null`，三題依序應是none、no、none；把已知的`false`誤寫成`null`會改變問題。對play而言，就緒未知時info仍可能為yes，Choice卻必須拒答。C-v3沒有把一般info的yes當成動作一定可採用的條件。這些規則用來製作訓練與檢查標籤；上線推論的答案由神經網路產生。

**明確欄位替模型分掉哪部分工作？** typed介面是有型別與欄位限制的介面：服務檢查名字、任務與布林值，再把裝置改名為內部A、B、C等代號，產生固定格式的材料與問題。任意長的使用者名字不必讓模型重新學習；名稱對應只比較ID，不查看哪個動作正確。這也限制了用途：新的任務或自由改寫的規則，不能只換文字就算已支援。

材料先切成裝置紀錄，每筆補齊到48個token；tokenizer優先使用學到的英文詞與中文字，未收錄片段再退回UTF-8 bytes，超長則拒絕而不截斷。每筆各用同一個Transformer編碼，筆內有位置、筆與筆之間沒有額外位置編號。全部結果接起來就是共享記憶體M，計算時還不知道後面會問哪個裝置。新問題才用公開的裝置ID選取M中對應的一筆，讓候選讀取它。這利用了輸入本來就明列「每筆屬於誰」的結構，減少把不同裝置欄位混在一起的工作；是否有許可、是否就緒與該給什麼答案，仍由已訓練的網路判斷。

下面是完整的縮小CPU算例，只示範Choice的新增分工。它使用隨機權重，維度縮為8、每筆4個token；正式成品是96維、每筆48個token。為直接看見索引，省去文字切詞，手填不同token代號；分筆編碼、候選獨立讀取與none聚合都放在程式裡。多頭注意力與Transformer本身沿用前面章節已教的PyTorch模組。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
torch.set_num_threads(1)
PAD = 0


class RowChoice(nn.Module):
    def __init__(self, vocabulary=64, width=8, row_tokens=4):
        super().__init__()
        self.width, self.row_tokens = width, row_tokens
        self.embedding = nn.Embedding(vocabulary, width, padding_idx=PAD)
        self.row_position = nn.Embedding(row_tokens, width)
        self.branch_position = nn.Embedding(16, width)
        self.state_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(width, 2, 3 * width, dropout=0, batch_first=True, norm_first=True),
            1,
            norm=nn.LayerNorm(width),
            enable_nested_tensor=False,
        )
        self.branch_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(width, 2, 3 * width, dropout=0, batch_first=True, norm_first=True),
            1,
            norm=nn.LayerNorm(width),
            enable_nested_tensor=False,
        )
        self.reader = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(width, 2, 3 * width, dropout=0, batch_first=True, norm_first=True),
            1,
            norm=nn.LayerNorm(width),
        )
        self.score = nn.Sequential(nn.Linear(width, width), nn.GELU(), nn.Linear(width, 1))
        self.none = nn.Sequential(nn.Linear(2 * width + 3, width), nn.GELU(), nn.Linear(width, 1))

    def material(self, states):
        # states: [batch, 裝置數 * 每筆tokens]；此時不接收問題。
        batch = len(states)
        rows = states.reshape(-1, self.row_tokens)
        positions = torch.arange(self.row_tokens)
        values = self.embedding(rows) + self.row_position(positions)
        encoded = self.state_encoder(values, src_key_padding_mask=rows.eq(PAD))
        return encoded.reshape(batch, -1, self.width)

    def forward(self, states, memory, target_rows, branches):
        # branches: [batch, 候選數, 問題+候選的tokens]，本例全為有效候選。
        batch, count, length = branches.shape
        indices = target_rows[:, None] * self.row_tokens + torch.arange(self.row_tokens)[None]
        selected_tokens = states.gather(1, indices)
        selected_memory = memory.gather(1, indices[..., None].expand(-1, -1, self.width))
        branch_tokens = branches.reshape(batch * count, length)
        query = self.embedding(branch_tokens) + self.branch_position(torch.arange(length))
        query = self.branch_encoder(query, src_key_padding_mask=branch_tokens.eq(PAD))
        read = self.reader(
            query,
            selected_memory.repeat_interleave(count, 0),
            tgt_key_padding_mask=branch_tokens.eq(PAD),
            memory_key_padding_mask=selected_tokens.eq(PAD).repeat_interleave(count, 0),
        )[:, 0]
        hidden = read.reshape(batch, count, self.width)
        scores = self.score(hidden).squeeze(-1)
        summary = torch.cat(
            (
                hidden.mean(1),
                hidden.max(1).values,
                scores.mean(1, keepdim=True),
                scores.max(1, keepdim=True).values,
                torch.full((batch, 1), count / 7),
            ),
            -1,
        )
        return torch.cat((scores, self.none(summary)), -1)


# 三筆各四格：1代表每筆／每個分支的起始符號，其餘代號只用來分辨輸入。
states = torch.tensor([[1, 10, 20, 30, 1, 11, 21, 31, 1, 12, 22, 32]])
names = ["A", "B", "C"]
target = "B"
target_rows = torch.tensor([names.index(target)])  # 只比名字，不讀許可或就緒。
branches = torch.tensor([[[1, 40, 50], [1, 40, 51], [1, 40, 52]]])
model = RowChoice().eval()
with torch.inference_mode():
    memory = model.material(states)
    scores = model(states, memory, target_rows, branches)
    permutation = torch.tensor([2, 0, 1])
    reordered = model(states, memory, target_rows, branches[:, permutation])
    moved_states = states.reshape(1, 3, 4)[:, [2, 0, 1]].reshape(1, -1)
    moved_memory = model.material(moved_states)
    moved_target = torch.tensor([["C", "A", "B"].index(target)])
    moved = model(moved_states, moved_memory, moved_target, branches)
print("M與分數形狀", tuple(memory.shape), tuple(scores.shape))
print("候選跟著重排", torch.allclose(reordered[:, :3], scores[:, permutation], atol=1e-5))
print("none不變", torch.allclose(reordered[:, -1], scores[:, -1], atol=1e-5))
print("搬動整筆仍查到B", torch.allclose(moved, scores, atol=1e-5))

輸出形狀為`(1, 12, 8)`與`(1, 4)`：12是三筆各四格，4是三個候選加none。後三列應全為True。分支前兩個token相同，最後一個不同，可看成同一問題搭配不同候選；`repeat_interleave`複製的是唯讀材料，各候選沒有讀其他候選的特徵。none才彙整平均、最大值、分數與數量，所以候選重排只改各自位置。整筆搬動後，ID索引也跟著改，仍讀到B原來的材料。以零起算的筆索引計，B原在第1筆（第二筆），取`1×4+[0,1,2,3]`，也就是第4–7格；搬後在第2筆，改取第8–11格。tokens與M都沿同一組indices取出，才不會一邊拿B、一邊拿別人的記憶。

這個算例省去了batch中不同數量的裝置／候選補齊、型別embedding、正式兩層材料編碼器和兩層reader；正式模型會遮住補齊項，info也會遮住none，rubric則保留缺欄時的none。算例沒有整筆空白紀錄，正式batch遇到這種補齊紀錄時會暫放起始符號，避免全遮罩的attention，再把該筆輸出清零。這些補齊不改上述分工。快取正式M時還會綁定材料、tokenizer及材料編碼器的指紋，避免換權重後誤用舊記憶體。算例的True只檢查運算關係，語義成績來自下面的已訓練成品。

**怎樣從分數決定是否採用？** 正式模型把分數除以該題型固定的溫度T，再算softmax。Choice先看模型是否選none、候選是否屬於目標裝置、最高機率是否達τ；對象檢查只比ID，不替它計算record或stop。下面對齊交付的C-v3服務，寫出選答與採用步驟；手填分數與τ只為展示規則，不是另一筆正式模型輸出。

前節的參考實作先比原scores，真正平手才比ID。C-v3服務目前直接對校準後機率取argmax：真正平手或浮點近似平手時，取列表中先出現的項，沒有沿用前節的固定ID規則。有兩項同為最高機率時，各項至多約0.5，無法通過Choice的0.99採用門檻；這個限制保護的是動作採用，被拒答的model_choice仍可能隨重排而換。info／rubric沒有Choice的採用門檻，也沒有另宣稱平手排序保障。下方8分差例驗的是採用與對象檢查，不是平手穩定性。


In [ ]:
def choose_with_policy(logits, ids, owners, target, temperature, threshold):
    probabilities = (logits / temperature).softmax(-1)
    index = int(probabilities.argmax())
    model_choice = ids[index]
    confidence = float(probabilities[index])
    accepted = model_choice != "none" and owners[index] == target and confidence >= threshold
    return model_choice, (model_choice if accepted else None), accepted


ids = ["record-main", "record-other", "none"]
owners = ["A", "B", None]
print(choose_with_policy(torch.tensor([8.0, 0.0, -2.0]), ids, owners, "A", 1.0, 0.99))
print(choose_with_policy(torch.tensor([0.0, 8.0, -2.0]), ids, owners, "A", 1.0, 0.99))

第一列採用`record-main`；第二列保留模型選了`record-other`的事實，但`selected`為None、採用為False。這能阻止建議用錯裝置，卻不能修好「同一裝置選錯動作」；模型原始選答正確率也把這類選錯對象的情況算錯，不會靠guard把錯誤藏掉。正式τ為0.99；Choice的T約0.458502，info與rubric各約0.420448。它們由另分的校準／政策資料固定，沒有看最後測試再調。info與rubric只提供判斷，不套用Choice的信心採用門檻；rubric選none時仍拒答。

**哪些工程改善需要重新驗收？** C-v3有630,443個不同參數，先由隨機參數訓練4,000次更新，再接自己的權重用新最佳化器更新1,500次。訓練材料包含8,000個裝置事實家族；同一事實改裝置名字、排列或題型，仍算同一家族，不能藉改寫混進不同切分。單一裝置有3種任務，許可與就緒各有是／否／未知三種，因此局部事實共有`3×3×3=27`種。

開發過程發現只問某個固定位置會漏掉其他裝置的狀態；後續在原訓練家族內輪流把不同裝置當目標，補齊27種局部事實，沒有把開發世界搬進訓練。訓練家族、開發、校準、政策與新最後測試按可見事實組合分開，改名與排列共用家族鍵；已看過的舊世界也先登記排除。這些同時改了資料、表示與訓練，不是只改資料量的單因子比較，也不能由兩版分數推論某一項改動單獨造成多少收益。

最後測試只在選好模型、溫度與政策後開一次。本次新的4–6裝置最後測試共1,500個事實家族：

| 檢查 | 實際結果 | 怎樣解讀 |
| --- | --- | --- |
| Choice／info／rubric | 每型各1,500／1,500 | 三型共用1,500家族，非4,500個獨立世界 |
| 可回答Choice的採用 | 1,025／1,025，錯誤0 | 可回答題的採用比例100% |
| 必須拒答的Choice | 475／475 | 362筆缺必要欄位、113筆移除正確候選 |
| 全部Choice的採用比例 | 1,025／1,500＝68.3% | 分母包含應拒答題，不等於漏接31.7%的可答題 |
| 手工局部事實檢查 | 三型各972／972，另移除候選432／432 | 涵蓋27種局部事實、3–6裝置的各位置及兩語；是工程檢查 |

在1,025筆已採用題裡零錯誤，仍不代表真正風險為零。把每筆視為同分布且獨立的錯／未錯二元觀察（伯努利觀察）時，單側95% Wilson近似上界約為0.263%；家族內三型與各種手工變體不能當成額外獨立樣本。這個數字也不保證新任務或任意自然語言的可靠度。三裝置另外通過手工表與原先已看過60個世界的新介面回歸，沒有另一份未見三裝置的正式測試，更沒有宣稱接受舊raw Question的自由文字API。

**直接使用已合格權重。** 在已依[W.1](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.1)安裝CPU課程環境、含`pyproject.toml`的repo根目錄執行：

```bash
uv run --extra cpu python -m scripts.course_product download
uv run --extra cpu python -m scripts.course_product judge
uv run --extra cpu python -m scripts.course_product judge --request examples/decision_product/record-unknown.json
uv run --extra cpu python -m scripts.course_product judge --request examples/decision_product/answer-absent.json
```

下載器按固定Release清單核對6,168,350-byte封包及四個推論檔，安裝到`checkpoints/decision-product-v3`，不執行封包程式；後三條只做本機推論。公開入口讀到`devices`就交給C-v3，回傳`product=C-v3`、`dispatch=typed-devices`；這是格式分派，沒有新神經路由。文字請求或不合法欄位會報錯，不會偷改派B或舊C。

第一份預設請求的實際CPU結果是Choice的`status=accepted`、`selected=record`，info的`selected=yes`、rubric的`selected=2`。第二份只把主錄音室的就緒改為未知，Choice與rubric的`status=abstained`、`selected=null`，info為no。第三份的候選缺少主錄音室record，雖有另一個裝置的record，Choice仍拒答。完整輸出還保留`model_choice/confidence/reason`；信心是固定校準後的分數，不會實際啟動裝置。

這三份是可操作範例，沒有另加進1,500個最後測試的分母。權重只有2,818,540 bytes，不代表整套Python／PyTorch安裝或執行記憶體也同樣小。CPU已核對；下載與操作不要求學生電腦載入Omni，GPU／MPS成績沒有由CPU結果建立。設定、tokenizer、固定政策、完整驗收、三項成品審閱與逐檔指紋在[固定配套與操作頁](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/decision-product/README.md)；這份ZIP用於推論與查證，沒有宣稱包含完整最佳化器／隨機狀態以精確續訓。

**練習：**只把主錄音室的permission改成false，保留ready=true。先按表格預測三題，再用自己的新JSON路徑核對。另把target換成播放台，說明「資訊完整」與「可以直接執行task」為何會是不同判斷；不要把模型輸出當成規則正解的來源。
